In [3]:
import glob
import os
import numpy as np
import pandas as pd

# ==========================================
# STUDENT-SPECIFIC CONFIGURATION
# ==========================================
SURNAME = "Solina"
STUDENT_ID = "TUPM-26-2302"

id_digits = [int(ch) for ch in STUDENT_ID if ch.isdigit()]
ID_SUM = sum(id_digits)                            # 15
SEED_DIGIT = id_digits[-1]                         # 2
RANDOM_SEED = int("".join(map(str, id_digits)))    # 262302

print(f"Student: {SURNAME} ({STUDENT_ID})")
print(f"ID_SUM = {ID_SUM} | SEED_DIGIT = {SEED_DIGIT} | RANDOM_SEED = {RANDOM_SEED}")

# ==========================================
# AUTO-DISCOVER SECOM DATASET IN PARENT / WORKSPACE FOLDERS
# ==========================================
def find_file(filename):
    search_roots = [".", "..", "../..", "../../.."]
    for root in search_roots:
        # Check direct path
        direct = os.path.join(root, filename)
        if os.path.exists(direct):
            return direct
        # Check immediate subdirectories (e.g., ../data/ or ../Lab7_.../)
        matches = glob.glob(os.path.join(root, "*", filename))
        if matches:
            return matches[0]
    return None

def load_secom_data():
    secom_data_path = find_file("secom.data")
    secom_labels_path = find_file("secom_labels.data")
    
    # Check for any CSV with 'secom' in the name
    csv_path = None
    for root in [".", "..", "../..", "../../.."]:
        matches = glob.glob(os.path.join(root, "secom.csv")) + glob.glob(os.path.join(root, "", "*secom.csv"))
        if matches:
            csv_path = matches[0]
            break

    if secom_data_path and secom_labels_path:
        print(f"Found raw SECOM files: {secom_data_path} & {secom_labels_path}")
        X_raw = np.loadtxt(secom_data_path)
        labels_df = pd.read_csv(
            secom_labels_path, sep=r"\s+", header=None, names=["label", "timestamp"]
        )
        y_raw = labels_df["label"].to_numpy()
        timestamps = labels_df["timestamp"].to_numpy()
        feature_names = [f"feature_{i}" for i in range(X_raw.shape[1])]
        return X_raw, y_raw, timestamps, feature_names

    elif csv_path:
        print(f"Found SECOM CSV: {csv_path}")
        df_raw = pd.read_csv(csv_path)
        timestamps = df_raw["Time"].to_numpy() if "Time" in df_raw.columns else np.arange(len(df_raw))
        label_col = "Pass/Fail" if "Pass/Fail" in df_raw.columns else df_raw.columns[-1]
        y_raw = df_raw[label_col].to_numpy()
        feature_cols = [c for c in df_raw.columns if c not in ["Time", label_col]]
        X_raw = df_raw[feature_cols].to_numpy(dtype=float)
        return X_raw, y_raw, timestamps, feature_cols

    else:
        raise FileNotFoundError(
            f"Could not find SECOM dataset near {os.getcwd()}. "
            "Check your main Lab 7 notebook for the exact dataset path."
        )

X, y, timestamps, feature_names = load_secom_data()
print(f"Loaded SECOM matrix: {X.shape[0]} records x {X.shape[1]} features")

Student: Solina (TUPM-26-2302)
ID_SUM = 15 | SEED_DIGIT = 2 | RANDOM_SEED = 262302
Found raw SECOM files: .\data\secom.data & .\data\secom_labels.data
Loaded SECOM matrix: 1567 records x 590 features


In [4]:
# ==========================================
# 1. VECTORIZED MEASUREMENT QUALITY ANALYSIS
# ==========================================

# 1. Identify features containing missing or constant observed values
has_missing = np.isnan(X).any(axis=0)
obs_min = np.nanmin(X, axis=0)
obs_max = np.nanmax(X, axis=0)
is_constant = np.isnan(obs_min) | (obs_min == obs_max)

# Usable features: no missing values and non-constant
usable_mask_ex1 = (~has_missing) & (~is_constant)
usable_indices_ex1 = np.where(usable_mask_ex1)[0]

# Also track all non-constant features if dataset requires features with missingness
# (Note: Standard SECOM has 590 features; ~538 have missing values, ~52 have 0 missing.
# If usable_indices_ex1 has fewer than needed, fallback to non-constant features.)
n_select_ex1 = 5 + (SEED_DIGIT % 3)
rng_ex1 = np.random.default_rng(RANDOM_SEED)

if len(usable_indices_ex1) >= n_select_ex1:
    selected_idx_ex1 = np.sort(rng_ex1.choice(usable_indices_ex1, size=n_select_ex1, replace=False))
else:
    non_constant_idx = np.where(~is_constant)[0]
    selected_idx_ex1 = np.sort(rng_ex1.choice(non_constant_idx, size=n_select_ex1, replace=False))

selected_features_ex1 = [feature_names[i] for i in selected_idx_ex1]
X_sub1 = X[:, selected_idx_ex1]

# 3. Calculate feature-wise statistics using vectorized NumPy
means_ex1 = np.nanmean(X_sub1, axis=0)
medians_ex1 = np.nanmedian(X_sub1, axis=0)
stds_ex1 = np.nanstd(X_sub1, axis=0, ddof=1)
missing_pct_ex1 = np.isnan(X_sub1).mean(axis=0) * 100.0

# 4. Compute robust standardized values using Median and MAD safely
abs_dev = np.abs(X_sub1 - medians_ex1)
mad_ex1 = np.nanmedian(abs_dev, axis=0)

# Safe division where MAD == 0
safe_mad = np.where(mad_ex1 == 0, np.nan, mad_ex1)
robust_z = np.where(mad_ex1 == 0, 0.0, (X_sub1 - medians_ex1) / safe_mad)

# 5. Identify measurements with |robust_z| > 3.5 (ignoring NaNs)
flagged_mask = np.abs(robust_z) > 3.5
flagged_counts = np.sum(flagged_mask, axis=0)
observed_counts = np.sum(~np.isnan(X_sub1), axis=0)
flagged_pct_ex1 = (flagged_counts / observed_counts) * 100.0

# Verify vectorized results against column-by-column loop
loop_flagged = np.array([
    np.sum(np.abs(np.where(mad_ex1[j] == 0, 0.0, (X_sub1[:, j] - medians_ex1[j]) / (mad_ex1[j] if mad_ex1[j] != 0 else 1.0))) > 3.5)
    for j in range(n_select_ex1)
])
vectorized_verified = bool(np.array_equal(flagged_counts, loop_flagged))

# 6. Rank features by percentage of flagged observations and export CSV
quality_df = pd.DataFrame({
    "Feature": selected_features_ex1,
    "Feature_Index": selected_idx_ex1,
    "Mean": means_ex1,
    "Median": medians_ex1,
    "Sample_Std": stds_ex1,
    "MAD": mad_ex1,
    "Missing_Percentage": missing_pct_ex1,
    "Flagged_Observations": flagged_counts,
    "Flagged_Percentage": flagged_pct_ex1,
})
quality_df = quality_df.sort_values(by="Flagged_Percentage", ascending=False).reset_index(drop=True)
quality_df["Rank"] = quality_df.index + 1
quality_df.to_csv("measurement_quality.csv", index=False)

# Assessment Data Output
print("=== EXERCISE 1: ASSESSMENT DATA ===")
print(f"Selected Features                 : {selected_features_ex1}")
print(f"Highest Missing Percentage        : {quality_df['Missing_Percentage'].max():.4f}%")
print(f"Feature with Highest Flagged %    : {quality_df.loc[0, 'Feature']} ({quality_df.loc[0, 'Flagged_Percentage']:.2f}%)")
print(f"Total Flagged Observations        : {int(quality_df['Flagged_Observations'].sum())}")
print(f"Vectorized Results Verified       : {vectorized_verified}")
display(quality_df)

=== EXERCISE 1: ASSESSMENT DATA ===
Selected Features                 : ['feature_249', 'feature_254', 'feature_361', 'feature_386', 'feature_493', 'feature_524', 'feature_576']
Highest Missing Percentage        : 0.0000%
Feature with Highest Flagged %    : feature_576 (6.70%)
Total Flagged Observations        : 253
Vectorized Results Verified       : True


,Feature,Feature_Index,Mean,Median,Sample_Std,MAD,Missing_Percentage,Flagged_Observations,Flagged_Percentage,Rank
0,feature_576,576,5.563747,1.6245,16.921369,0.2496,0.0,105,6.700702,1
1,feature_386,386,0.008281,0.0068,0.015488,0.0022,0.0,84,5.360562,2
2,feature_361,361,39.936406,39.6961,17.056304,7.5138,0.0,32,2.042119,3
3,feature_524,524,5.687782,4.8771,20.663414,1.2997,0.0,16,1.021059,4
4,feature_493,493,2.530046,2.5291,0.973948,0.7594,0.0,12,0.765795,5
5,feature_254,254,0.013943,0.0150,0.009132,0.0090,0.0,4,0.255265,6
6,feature_249,249,0.001065,0.0000,0.015771,0.0000,0.0,0,0.000000,7


In [5]:
# ==========================================
# 2. MOBILE DEVICE USAGE PROFILING
# ==========================================

# 1. Construct labeled DataFrame preserving original SECOM record identifiers
df_secom = pd.DataFrame(X, columns=feature_names)
df_secom.insert(0, "record_id", np.arange(len(df_secom)))
df_secom.insert(1, "timestamp", timestamps)
df_secom.insert(2, "outcome", np.where(y == 1, "Fail", "Pass"))
df_secom.insert(3, "label", y)

# 2. Create reproducible balanced sample: min(20 + ID_SUM, Pass count, Fail count) per outcome
pass_df = df_secom[df_secom["outcome"] == "Pass"]
fail_df = df_secom[df_secom["outcome"] == "Fail"]

n_per_group = min(20 + ID_SUM, len(pass_df), len(fail_df))
rng_ex2 = np.random.default_rng(RANDOM_SEED + 2)

pass_sample = pass_df.sample(n=n_per_group, random_state=int(rng_ex2.integers(0, 1_000_000)))
fail_sample = fail_df.sample(n=n_per_group, random_state=int(rng_ex2.integers(0, 1_000_000)))
balanced_df = pd.concat([pass_sample, fail_sample], axis=0).sort_values("record_id").reset_index(drop=True)

# 3. Select 5 usable measurement features & apply median imputation from full dataset
usable_features_ex2 = [
    feature_names[i] for i in range(len(feature_names))
    if not is_constant[i] and df_secom[feature_names[i]].notna().sum() >= 10
]
selected_features_ex2 = sorted(
    list(rng_ex2.choice(usable_features_ex2, size=5, replace=False)),
    key=lambda name: feature_names.index(name)
)

full_medians = df_secom[selected_features_ex2].median()
balanced_imputed = balanced_df[["record_id", "timestamp", "outcome", "label"] + selected_features_ex2].copy()
balanced_imputed[selected_features_ex2] = balanced_imputed[selected_features_ex2].fillna(full_medians)

# Export balanced_sample.csv
balanced_imputed.to_csv("balanced_sample.csv", index=False)

# 4. Use groupby() and agg() for counts, means, medians, and sample standard deviations
grouped_stats = balanced_imputed.groupby("outcome")[selected_features_ex2].agg(
    ["count", "mean", "median", lambda s: s.std(ddof=1)]
)
grouped_stats.columns = [
    f"{feat}_{stat if stat != '<lambda_0>' else 'std'}"
    for feat, stat in grouped_stats.columns
]

# 5. Calculate Absolute Mean Difference and Standardized Mean Difference (Cohen's d)
comparison_rows = []
numpy_pandas_match = True

for feat in selected_features_ex2:
    pass_vals = balanced_imputed.loc[balanced_imputed["outcome"] == "Pass", feat].to_numpy()
    fail_vals = balanced_imputed.loc[balanced_imputed["outcome"] == "Fail", feat].to_numpy()

    n_pass, n_fail = grouped_stats.loc["Pass", f"{feat}_count"], grouped_stats.loc["Fail", f"{feat}_count"]
    mean_pass, mean_fail = grouped_stats.loc["Pass", f"{feat}_mean"], grouped_stats.loc["Fail", f"{feat}_mean"]
    med_pass, med_fail = grouped_stats.loc["Pass", f"{feat}_median"], grouped_stats.loc["Fail", f"{feat}_median"]
    std_pass, std_fail = grouped_stats.loc["Pass", f"{feat}_std"], grouped_stats.loc["Fail", f"{feat}_std"]

    # Verify NumPy vs Pandas statistics match
    if not (
        np.isclose(mean_pass, np.mean(pass_vals))
        and np.isclose(mean_fail, np.mean(fail_vals))
        and np.isclose(std_pass, np.std(pass_vals, ddof=1))
        and np.isclose(std_fail, np.std(fail_vals, ddof=1))
    ):
        numpy_pandas_match = False

    abs_mean_diff = abs(mean_fail - mean_pass)
    pooled_var = (((n_pass - 1) * (std_pass * 2)) + ((n_fail - 1) * (std_fail * 2))) / (n_pass + n_fail - 2)
    pooled_std = np.sqrt(pooled_var)

    # Safe division when pooled_std == 0
    abs_smd = 0.0 if pooled_std == 0 else abs_mean_diff / pooled_std

    comparison_rows.append({
        "Feature": feat,
        "Pass_Count": int(n_pass),
        "Fail_Count": int(n_fail),
        "Pass_Mean": mean_pass,
        "Fail_Mean": mean_fail,
        "Pass_Median": med_pass,
        "Fail_Median": med_fail,
        "Pass_Std": std_pass,
        "Fail_Std": std_fail,
        "Pooled_Std": pooled_std,
        "Abs_Mean_Diff": abs_mean_diff,
        "Abs_Standardized_Mean_Diff": abs_smd,
    })

# 6. Rank by Absolute Standardized Mean Difference and export CSV
feature_comp_df = pd.DataFrame(comparison_rows).sort_values(
    by="Abs_Standardized_Mean_Diff", ascending=False
).reset_index(drop=True)
feature_comp_df["Rank"] = feature_comp_df.index + 1
feature_comp_df.to_csv("feature_comparison.csv", index=False)

# Assessment Data Output
print("=== EXERCISE 2: ASSESSMENT DATA ===")
print(f"Balanced Sample Size              : {len(balanced_imputed)} ({n_per_group} Pass, {n_per_group} Fail)")
print(f"Selected Features                 : {selected_features_ex2}")
print(f"Highest-Ranked Feature            : {feature_comp_df.loc[0, 'Feature']}")
print(f"Absolute Standardized Mean Diff   : {feature_comp_df.loc[0, 'Abs_Standardized_Mean_Diff']:.4f}")
print(f"NumPy-Pandas Statistics Match     : {numpy_pandas_match}")
display(feature_comp_df)

=== EXERCISE 2: ASSESSMENT DATA ===
Balanced Sample Size              : 70 (35 Pass, 35 Fail)
Selected Features                 : [np.str_('feature_71'), np.str_('feature_150'), np.str_('feature_168'), np.str_('feature_342'), np.str_('feature_410')]
Highest-Ranked Feature            : feature_410
Absolute Standardized Mean Diff   : 0.3987
NumPy-Pandas Statistics Match     : True


,Feature,Pass_Count,Fail_Count,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Std,Fail_Std,Pooled_Std,Abs_Mean_Diff,Abs_Standardized_Mean_Diff,Rank
0,feature_410,35,35,4.274834,5.064380,4.0073,5.3351,1.936792,1.984028,1.980106,0.789546,0.398739,1
1,feature_150,35,35,6.865229,6.642229,6.2360,5.8910,2.808225,3.448988,2.501442,0.223000,0.089149,2
2,feature_168,35,35,0.133200,0.117429,0.1220,0.1170,0.040299,0.038051,0.279911,0.015771,0.056344,3
3,feature_71,35,35,100.067154,100.430257,97.2656,103.7037,21.718491,31.882949,7.321300,0.363103,0.049595,4
4,feature_342,35,35,0.000000,0.000000,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,5


In [6]:
# ==========================================
# 3. STATISTICAL ROBUSTNESS & SENSITIVITY ANALYSIS
# ==========================================

# 1. Reproducibly select 5 usable features with >= 10 observed measurements in each outcome group
pass_mask = (df_secom["outcome"] == "Pass").to_numpy()
fail_mask = (df_secom["outcome"] == "Fail").to_numpy()

eligible_ex3 = [
    feature_names[i] for i in range(len(feature_names))
    if (not is_constant[i])
    and (np.sum(~np.isnan(X[pass_mask, i])) >= 10)
    and (np.sum(~np.isnan(X[fail_mask, i])) >= 10)
]

rng_ex3_select = np.random.default_rng(RANDOM_SEED + 3)
selected_features_ex3 = sorted(
    list(rng_ex3_select.choice(eligible_ex3, size=5, replace=False)),
    key=lambda name: feature_names.index(name)
)

n_boot = 200 + 10 * SEED_DIGIT

def run_sensitivity_pipeline(seed_value):
    rng = np.random.default_rng(seed_value)
    results = []

    for feat in selected_features_ex3:
        # Strategy A: Available-Case Analysis
        pass_avail = df_secom.loc[pass_mask, feat].dropna().to_numpy()
        fail_avail = df_secom.loc[fail_mask, feat].dropna().to_numpy()
        obs_diff_avail = np.mean(fail_avail) - np.mean(pass_avail)

        # Strategy B: Full-Dataset Median Imputation
        feat_median = df_secom[feat].median()
        pass_imp = df_secom.loc[pass_mask, feat].fillna(feat_median).to_numpy()
        fail_imp = df_secom.loc[fail_mask, feat].fillna(feat_median).to_numpy()
        obs_diff_imp = np.mean(fail_imp) - np.mean(pass_imp)

        # Stratified Bootstrap Resampling (Available-Case)
        boot_avail_pass = rng.choice(pass_avail, size=(n_boot, len(pass_avail)), replace=True).mean(axis=1)
        boot_avail_fail = rng.choice(fail_avail, size=(n_boot, len(fail_avail)), replace=True).mean(axis=1)
        boot_diffs_avail = boot_avail_fail - boot_avail_pass

        # Stratified Bootstrap Resampling (Median Imputed)
        boot_imp_pass = rng.choice(pass_imp, size=(n_boot, len(pass_imp)), replace=True).mean(axis=1)
        boot_imp_fail = rng.choice(fail_imp, size=(n_boot, len(fail_imp)), replace=True).mean(axis=1)
        boot_diffs_imp = boot_imp_fail - boot_imp_pass

        results.append({
            "Feature": feat,
            "Avail_Mean_Diff": obs_diff_avail,
            "Avail_Boot_Mean_Diff": np.mean(boot_diffs_avail),
            "Avail_CI_Lower_95": np.percentile(boot_diffs_avail, 2.5),
            "Avail_CI_Upper_95": np.percentile(boot_diffs_avail, 97.5),
            "Imputed_Mean_Diff": obs_diff_imp,
            "Imputed_Boot_Mean_Diff": np.mean(boot_diffs_imp),
            "Imputed_CI_Lower_95": np.percentile(boot_diffs_imp, 2.5),
            "Imputed_CI_Upper_95": np.percentile(boot_diffs_imp, 97.5),
            "Abs_Change_Mean_Diff": abs(obs_diff_imp - obs_diff_avail),
            "Sign_Changed": bool(np.sign(obs_diff_avail) != np.sign(obs_diff_imp)),
        })

    df_res = pd.DataFrame(results)
    # Rank by absolute mean difference under each strategy
    df_res["Avail_Rank"] = df_res["Avail_Mean_Diff"].abs().rank(ascending=False).astype(int)
    df_res["Imputed_Rank"] = df_res["Imputed_Mean_Diff"].abs().rank(ascending=False).astype(int)
    df_res["Ranking_Changed"] = df_res["Avail_Rank"] != df_res["Imputed_Rank"]
    return df_res

# Execute pipeline twice with the same seed to verify reproducibility
robustness_df = run_sensitivity_pipeline(RANDOM_SEED + 300)
robustness_df_check = run_sensitivity_pipeline(RANDOM_SEED + 300)
reproducibility_verified = bool(robustness_df.equals(robustness_df_check))

# Export robustness_comparison.csv
robustness_df.to_csv("robustness_comparison.csv", index=False)

# Extract summary metrics
largest_change = robustness_df["Abs_Change_Mean_Diff"].max()
changed_sign_feats = robustness_df.loc[robustness_df["Sign_Changed"], "Feature"].tolist()
changed_rank_feats = robustness_df.loc[robustness_df["Ranking_Changed"], "Feature"].tolist()

# Export sensitivity_report.txt
report_lines = [
    "STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT",
    f"Student: {SURNAME} ({STUDENT_ID})",
    f"Random Seed: {RANDOM_SEED + 300} | Bootstrap Repetitions: {n_boot}",
    "-" * 60,
    f"Selected Features: {', '.join(selected_features_ex3)}",
    f"Largest Change in Mean Difference: {largest_change:.6f}",
    f"Features with Changed Difference Sign: {changed_sign_feats if changed_sign_feats else 'None'}",
    f"Features with Changed Ranking: {changed_rank_feats if changed_rank_feats else 'None'}",
    f"Reproducibility Verified: {reproducibility_verified}",
    "-" * 60,
    "Interpretation:",
    "1. Comparing available-case analysis against full-dataset median imputation evaluates",
    "   whether missingness introduces bias into Fail-vs-Pass group mean differences.",
    "2. Percentile 95% confidence intervals that exclude zero indicate statistically stable",
    "   group differences under stratified bootstrap resampling.",
    "3. Any shift in sign or feature importance ranking highlights sensitivity to missing-data",
    "   handling and warrants cautious engineering interpretation."
]

with open("sensitivity_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

# Assessment Data Output
print("=== EXERCISE 3: ASSESSMENT DATA ===")
print(f"Selected Features                 : {selected_features_ex3}")
print(f"Bootstrap Repetitions             : {n_boot}")
print(f"Largest Change in Mean Difference : {largest_change:.6f}")
print(f"Features with Changed Diff Sign   : {changed_sign_feats if changed_sign_feats else 'None'}")
print(f"Features with Changed Ranking     : {changed_rank_feats if changed_rank_feats else 'None'}")
print(f"Reproducibility Verified          : {reproducibility_verified}")
display(robustness_df)

=== EXERCISE 3: ASSESSMENT DATA ===
Selected Features                 : [np.str_('feature_118'), np.str_('feature_207'), np.str_('feature_217'), np.str_('feature_270'), np.str_('feature_301')]
Bootstrap Repetitions             : 220
Largest Change in Mean Difference : 0.002013
Features with Changed Diff Sign   : None
Features with Changed Ranking     : None
Reproducibility Verified          : True


,Feature,Avail_Mean_Diff,Avail_Boot_Mean_Diff,Avail_CI_Lower_95,Avail_CI_Upper_95,Imputed_Mean_Diff,Imputed_Boot_Mean_Diff,Imputed_CI_Lower_95,Imputed_CI_Upper_95,Abs_Change_Mean_Diff,Sign_Changed,Avail_Rank,Imputed_Rank,Ranking_Changed
0,feature_118,-0.000739,-0.000740,-0.002316,0.000428,-0.000748,-0.000807,-0.002027,0.000557,0.000009,False,5,5,False
1,feature_207,2.481148,2.546053,-0.971459,7.471881,2.483161,2.533954,-1.008542,8.769380,0.002013,False,1,1,False
2,feature_217,0.002138,0.001893,-0.004839,0.010893,0.002298,0.002518,-0.004580,0.010265,0.000161,False,4,4,False
3,feature_270,-0.952016,-1.014589,-2.765270,0.866453,-0.950136,-1.006272,-2.491235,1.149281,0.001880,False,2,2,False
4,feature_301,0.064112,0.058099,-0.028685,0.157636,0.063089,0.059496,-0.029795,0.176679,0.001023,False,3,3,False
